<a href="https://colab.research.google.com/github/minhvu2105/alpha-evm-dex-bot/blob/main/GPT_SoVITS_Colab_VI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-SoVITS v2 - Vietnamese Voice Cloning Pipeline (Optimized)

Notebook này đã được tối ưu hóa để linh hoạt và dễ sử dụng hơn.

### 🚀 Quy trình thực hiện:
1. **Cài đặt môi trường**: Chạy 1 lần đầu tiên.
2. **Cấu hình chung**: Đặt tên model (Experiment Name) tại đây.
3. **Tải Dữ liệu**: Upload file âm thanh từ máy hoặc Drive.
4. **Xử lý Dữ liệu**: Cắt âm thanh và tạo phụ đề (ASR).
5. **WebUI**: Mở giao diện để Train và Inference.

In [1]:
# @title 1. Khởi tạo Môi trường, Fix OpenCC & Vá lỗi Logic (All-in-One)
import os
from google.colab import drive
from IPython.display import clear_output

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive')
    except:
        print('Đã mount Drive hoặc đang dùng Local Runtime.')

# 2. Clone/Update Repo
%cd /content
if not os.path.exists("GPT-SoVITS-Vietnamese"):
    print("🚀 Đang clone repository...")
    !git clone https://github.com/tqtuan8788-ai/GPT-SoVITS-Vietnamese.git
%cd GPT-SoVITS-Vietnamese

# 3. Cài đặt Thư viện Hệ thống (Fix OpenCC Build Error)
print("🛠️ Cài đặt thư viện hệ thống và fix build OpenCC...")
!apt-get update
!apt-get install -y libopencc-dev libsndfile1 ffmpeg cmake build-essential

# 4. Cài đặt Python Packages (Ưu tiên bản build sẵn để tránh lỗi Wheel)
print("📦 Đang cài đặt danh sách thư viện tùy chỉnh...")
# Cài đặt các công cụ build trước
!pip install -q setuptools wheel
# Cài đặt OpenCC bản reimplemented (ổn định nhất trên Colab)
!pip install -q opencc-python-reimplemented
# Cài đặt các thư viện lõi
!pip install -q "numpy<2.0" scipy tensorboard "librosa==0.10.2" numba "pytorch-lightning>=2.4" "gradio<5" ffmpeg-python tqdm "funasr==1.0.27" cn2an pypinyin "pyopenjtalk>=0.4.1" g2p_en torchaudio modelscope sentencepiece "transformers>=4.43,<=4.50" "peft<0.18.0" chardet PyYAML psutil jieba_fast jieba split-lang "fast_langdetect>=0.3.1" wordsegment rotary_embedding_torch ToJyutping g2pk2 ko_pron python_mecab_ko "fastapi[standard]>=0.115.2" x_transformers "torchmetrics<=1.5" "pydantic<=2.10.6" "ctranslate2>=4.0,<5" "av>=11" onnxruntime-gpu
!pip install -q faster-whisper

# 5. FIX TRIỆT ĐỂ: Vá lỗi Unpickling (PyTorch 2.6) & Lỗi chia cho 0 (Ít file)
print("🔧 Đang áp dụng các bản vá logic vào source code...")
patches = {
    "GPT_SoVITS/prepare_datasets/2-get-sv.py": [
        ('torch.load(sv_path, map_location="cpu")', 'torch.load(sv_path, map_location="cpu", weights_only=False)')
    ],
    "GPT_SoVITS/AR/data/data_module.py": [
        ('torch.load(dict_s1, map_location="cpu")', 'torch.load(dict_s1, map_location="cpu", weights_only=False)')
    ],
    "GPT_SoVITS/AR/data/dataset.py": [
        ('if(norm_text_len/wav_dur<3 or norm_text_len/wav_dur>25):continue', 'if(norm_text_len/wav_dur<0.01 or norm_text_len/wav_dur>100):continue'),
        ('int(min_num / leng)', 'int(min_num / max(leng, 1))'),
        ('assert len(audiopaths_sid_text_new) > 1', 'assert len(audiopaths_sid_text_new) >= 1')
    ],
    "tools/my_utils.py": [
        ('cmd=[r"C:\\Users\\Administrator\\Desktop\\v7.1\\ffmpeg-win-x86_64-v7.1.exe", "-nostdin"]', 'cmd=["ffmpeg", "-nostdin"]')
    ]
}

for fpath, changes in patches.items():
    full_path = os.path.join("/content/GPT-SoVITS-Vietnamese", fpath)
    if os.path.exists(full_path):
        with open(full_path, 'r') as f:
            content = f.read()
        for old, new in changes:
            content = content.replace(old, new)
        with open(full_path, 'w') as f:
            f.write(content)
        print(f"✅ Đã vá: {fpath}")

clear_output()
print("🎯 MÔI TRƯỜNG ĐÃ SẴN SÀNG & ĐÃ FIX LỖI OPENCC!")
print("👉 Bây giờ bạn hãy tiếp tục chạy Part 2 để tải Model sạch nhé.")

🎯 MÔI TRƯỜNG ĐÃ SẴN SÀNG & ĐÃ FIX LỖI OPENCC!
👉 Bây giờ bạn hãy tiếp tục chạy Part 2 để tải Model sạch nhé.


In [6]:
# @title 2. Cấu hình Toàn diện & Đồng bộ Hệ thống với Drive
exp_name = "Giong_Doc_Sach_02" # @param {type:"string"}

import os
# Đường dẫn gốc trên Drive
drive_base = f"/content/drive/MyDrive/GPT_SoVITS_Data"
project_path = f"{drive_base}/{exp_name}"
pretrain_path = f"{drive_base}/PRETRAINED_MODELS_BASE"

# 1. Tạo cấu trúc folder trên Drive (Đầy đủ folder Audio và Config)
folders = [
    f"{project_path}/weights",       # Lưu model sau khi train
    f"{project_path}/logs",          # Lưu log training
    f"{project_path}/sliced_audio",  # Lưu audio đã cắt
    f"{project_path}/raw_audio",     # Lưu audio gốc bạn upload
    pretrain_path                    # Lưu bộ model gốc (Pretrained)
]
for f in folders: os.makedirs(f, exist_ok=True)

# 2. Xử lý Symlinks (Xóa folder ảo cũ, thay bằng folder Drive)
# --- Link Weights & Logs ---
!rm -rf /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/weights
!ln -s {project_path}/weights /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/weights
!mkdir -p /content/GPT-SoVITS-Vietnamese/logs
!rm -rf /content/GPT-SoVITS-Vietnamese/logs/{exp_name}
!ln -s {project_path}/logs /content/GPT-SoVITS-Vietnamese/logs/{exp_name}

# --- Link Pretrained Models (Đây là chìa khóa để tải thẳng vào Drive) ---
target_pretrain = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models"
!rm -rf {target_pretrain}
!ln -s {pretrain_path} {target_pretrain}

print(f"✅ Hệ thống đã sẵn sàng cho Project: {exp_name}")
print(f"📦 Toàn bộ dữ liệu & Pretrain sẽ được đồng bộ tại: {drive_base}")

✅ Hệ thống đã sẵn sàng cho Project: Giong_Doc_Sach_02
📦 Toàn bộ dữ liệu & Pretrain sẽ được đồng bộ tại: /content/drive/MyDrive/GPT_SoVITS_Data


In [14]:
# @title 3. Tải Pretrained Models (Bản Full cấu hình & Fix lỗi HuBERT/SV)
import os

base_model_dir = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models"
hubert_dir = f"{base_model_dir}/chinese-hubert-base"
roberta_dir = f"{base_model_dir}/chinese-roberta-wwm-ext-large"
sv_dir = f"{base_model_dir}/sv"
v2_dir = f"{base_model_dir}/v2Pro"

if not os.path.islink(base_model_dir):
    print("⚠️ Cảnh báo: Bạn cần chạy Part 3 trước để kết nối Drive!")
else:
    # Tạo folder nếu chưa có
    for d in [hubert_dir, roberta_dir, sv_dir, v2_dir]:
        os.makedirs(d, exist_ok=True)

    print("📥 1. Đang tải trọn bộ RoBERTa (Fix lỗi Tokenizer)...")
    !wget -nc -P {roberta_dir} https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/pytorch_model.bin
    !wget -nc -P {roberta_dir} https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/config.json
    !wget -nc -P {roberta_dir} https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/vocab.txt
    !wget -nc -P {roberta_dir} https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/tokenizer.json
    !wget -nc -P {roberta_dir} https://huggingface.co/hfl/chinese-roberta-wwm-ext-large/resolve/main/added_tokens.json

    print("📥 2. Đang tải HuBERT & Cấu hình (Fix lỗi OSError ở Tab 1A)...")
    !wget -nc -P {hubert_dir} https://huggingface.co/TencentGameMate/chinese-hubert-base/resolve/main/pytorch_model.bin
    !wget -nc -P {hubert_dir} https://huggingface.co/TencentGameMate/chinese-hubert-base/resolve/main/config.json
    !wget -nc -P {hubert_dir} https://huggingface.co/TencentGameMate/chinese-hubert-base/resolve/main/preprocessor_config.json

    print("📥 3. Đang tải SV & v2Pro (Fix lỗi EOFError)...")
    !wget -nc -O {sv_dir}/pretrained_eres2netv2w24s4ep4.ckpt https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/pretrained_models/sv/pretrained_eres2netv2w24s4ep4.ckpt
    !wget -nc -O {v2_dir}/s2Gv2Pro.pth https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/v2Pro/s2Gv2Pro.pth
    !wget -nc -O {v2_dir}/s2Dv2Pro.pth https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/v2Pro/s2Dv2Pro.pth
    !wget -nc -O {base_model_dir}/s1v3.ckpt https://huggingface.co/lj1995/GPT-SoVITS/resolve/main/s1v3.ckpt

    print("\n✅ TẤT CẢ MODEL ĐÃ SẴN SÀNG TRÊN DRIVE!")

📥 1. Đang tải trọn bộ RoBERTa (Fix lỗi Tokenizer)...
File ‘/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/pytorch_model.bin’ already there; not retrieving.

File ‘/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/config.json’ already there; not retrieving.

File ‘/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/vocab.txt’ already there; not retrieving.

File ‘/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/tokenizer.json’ already there; not retrieving.

File ‘/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large/added_tokens.json’ already there; not retrieving.

📥 2. Đang tải HuBERT & Cấu hình (Fix lỗi OSError ở Tab 1A)...
File ‘/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-hubert-base/pytorch_model.bin’ already there; not retrieving.

--2026-05-08 03:33:23--  https

1. Vị trí đặt Audio và file .list
Bạn hãy truy cập vào Google Drive của mình và tìm đến thư mục theo đúng lộ trình sau:


File Audio đã cắt (.wav): Bỏ vào thư mục /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/sliced_audio/.  


File danh sách kịch bản (.list): Bỏ trực tiếp vào thư mục gốc của dự án: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/Giong_Doc_Sach_02.list.

In [16]:
# @title 🛠️ Fix FFmpeg Path Error (Linux vs Windows)
import os

# Đường dẫn đến file bị lỗi
file_to_fix = "/content/GPT-SoVITS-Vietnamese/tools/my_utils.py"

if os.path.exists(file_to_fix):
    with open(file_to_fix, 'r', encoding='utf-8') as f:
        content = f.read()

    # Thay thế đường dẫn Windows rác bằng lệnh ffmpeg chuẩn của Linux
    wrong_path = r"C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\imageio_ffmpeg\binaries\ffmpeg-win-x86_64-v7.1.exe"
    fixed_content = content.replace(wrong_path, "ffmpeg")

    with open(file_to_fix, 'w', encoding='utf-8') as f:
        f.write(fixed_content)

    print("✅ Đã vá lỗi đường dẫn FFmpeg thành công!")
else:
    print("❌ Không tìm thấy file cần vá. Hãy kiểm tra lại repo.")

# Cài đặt lại ffmpeg hệ thống cho chắc chắn
!apt-get install -y ffmpeg

✅ Đã vá lỗi đường dẫn FFmpeg thành công!
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 132 not upgraded.


In [24]:
# @title 🛠️ Fix Path: Tự động kết nối thư mục PRETRAINED_MODELS
import os

# 1. Tự động tìm thư mục của Vũ trên Drive (chấp nhận cả viết hoa/thường)
drive_root = "/content/drive/MyDrive/GPT_SoVITS_Data"
possible_names = [f for f in os.listdir(drive_root) if f.upper().startswith("PRETRAINED_MODELS")]

if not possible_names:
    print("❌ Vẫn không thấy thư mục PRETRAINED trên Drive. Vũ kiểm tra lại tên nhé!")
else:
    real_drive_folder = os.path.join(drive_root, possible_names[0])
    system_base = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models"

    print(f"📂 Đã tìm thấy thư mục thực tế: {possible_names[0]}")

    # Danh sách file cần khớp
    check_list = {
        "sv": "pretrained_eres2netv2w24s4ep4.ckpt",
        "v2Pro": "s2Gv2Pro.pth",
        "v2Pro": "s2Dv2Pro.pth"
    }

    for sub, f_name in check_list.items():
        src = os.path.join(real_drive_folder, sub, f_name)
        # Riêng file trong v2Pro đôi khi tác giả đặt tên khác, ta linh hoạt kiểm tra
        if not os.path.exists(src) and sub == "v2Pro":
             # Thử tìm file có đuôi .pth trong thư mục đó
             files = os.listdir(os.path.join(real_drive_folder, sub))
             if files: src = os.path.join(real_drive_folder, sub, files[0])

        dst_folder = os.path.join(system_base, sub)
        os.makedirs(dst_folder, exist_ok=True)
        dst = os.path.join(dst_folder, f_name)

        if os.path.exists(src):
            if os.path.exists(dst): os.remove(dst)
            os.symlink(src, dst)
            print(f"✅ Đã kết nối thành công: {f_name}")
        else:
            print(f"❌ Vẫn thiếu file: {sub}/{f_name}")

print("\n🚀 Vũ hãy quay lại WebUI và nhấn 'Start formatting' ở Tab 1A ngay nhé!")

📂 Đã tìm thấy thư mục thực tế: PRETRAINED_MODELS_BASE
✅ Đã kết nối thành công: pretrained_eres2netv2w24s4ep4.ckpt
✅ Đã kết nối thành công: s2Dv2Pro.pth

🚀 Vũ hãy quay lại WebUI và nhấn 'Start formatting' ở Tab 1A ngay nhé!


In [ ]:
# @title 4. Khởi động WebUI (Bản chuẩn hóa cho Minh Vũ)
import os

# 1. Đồng bộ tên dự án
if 'exp_name' not in locals():
    exp_name = "Giong_Doc_Sach_02"

%cd /content/GPT-SoVITS-Vietnamese

# 2. Cấu hình đường dẫn Drive (Tự động theo Part 3)
drive_base = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"
asr_output_file = f"{drive_base}/{exp_name}.list"
sliced_folder = f"{drive_base}/sliced_audio"

# 3. Thiết lập môi trường (Ưu tiên dùng Symlink từ Drive)
os.environ["cnhubert_base_path"] = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-hubert-base"
os.environ["bert_path"] = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large"
os.environ["colab_active"] = "1"

# 4. Ép cấu hình Share
!sed -i "s/is_share = False/is_share = True/g" config.py

print("="*60)
print(f"📋 THÔNG SỐ COPY VÀO TAB 1A & 1B:")
print(f"🔹 Experiment Name: {exp_name}")
print(f"🔹 Text Label File: {asr_output_file}")
print(f"🔹 Dataset Folder: {sliced_folder}")
print("="*60)

# 5. Chạy WebUI với cờ share cưỡng bức
!python webui.py --share

/content/GPT-SoVITS-Vietnamese
📋 THÔNG SỐ COPY VÀO TAB 1A & 1B:
🔹 Experiment Name: Giong_Doc_Sach_02
🔹 Text Label File: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/Giong_Doc_Sach_02.list
🔹 Dataset Folder: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/sliced_audio
Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://9a977c8961dd0fe7d0.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [ ]:
# @title ⚙️ 2. Kiểm tra Trạng thái Dữ liệu
import os

# Kiểm tra xem Bước 1 đã chạy chưa
if 'global_exp_name' not in globals():
    print("❌ Lỗi: Bạn cần chạy Bước 1 trước để thiết lập tên Experiment!")
else:
    exp_name = global_exp_name
    drive_save_path = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"

    print(f"✅ Đang làm việc với Project: {exp_name}")

    # Kiểm tra xem có model cũ không để nhắc Resume
    if os.path.exists(f"{drive_save_path}/weights"):
        files = [f for f in os.listdir(f"{drive_save_path}/weights") if f.endswith(('.pth', '.ckpt'))]
        if files:
            print(f"🔄 Tìm thấy {len(files)} file model cũ trên Drive. Bạn có thể Train tiếp (Resume).")
        else:
            print("📝 Thư mục Drive sạch sẽ, sẵn sàng Train mới.")

    # Thiết lập đường dẫn cho các bước sau
    dataset_root = "/content/dataset"
    output_root = "/content/GPT-SoVITS-Vietnamese/output"

    print(f"\n👉 Tiếp theo: Hãy upload Audio vào thư mục {dataset_root}/{exp_name}")

✅ Đang làm việc với Project: Giong_Doc_Sach_02
📝 Thư mục Drive sạch sẽ, sẵn sàng Train mới.

👉 Tiếp theo: Hãy upload Audio vào thư mục /content/dataset/Giong_Doc_Sach_02


In [13]:
# @title 5. Khởi động WebUI (Gradio)
# @markdown Cell này sẽ chạy liên tục để duy trì giao diện. Hãy click vào link `gradio.live` hiện ra ở dưới.

import os

# 1. Kiểm tra và lấy tên Experiment từ Bước 1
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_02"
    print(f"⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: {exp_name}")
else:
    exp_name = global_exp_name

%cd /content/GPT-SoVITS-Vietnamese

# 2. Cấu hình đường dẫn cho WebUI (Tự động nhận diện v2Pro)
cwd = os.getcwd()
drive_base = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"
asr_output_file = f"{drive_base}/{exp_name}.list"
sliced_folder = f"{drive_base}/sliced_audio"

# Thiết lập biến môi trường
os.environ["cnhubert_base_path"] = os.path.join(cwd, "GPT_SoVITS/pretrained_models/chinese-hubert-base")
os.environ["bert_path"] = os.path.join(cwd, "GPT_SoVITS/pretrained_models/chinese-roberta-wwm-ext-large")
os.environ["colab_active"] = "1"
os.environ["is_share"] = "True"

# 3. Thông tin hướng dẫn nhanh
print("="*60)
print("🚀 ĐANG KHỞI ĐỘNG GPT-SOVITS WebUI...")
print("="*60)
print(f"📋 THÔNG SỐ ĐỂ ĐIỀN VÀO WEBUI (TAB 1A & 1B):")
print(f"🔹 Experiment Name: {exp_name}")
print(f"🔹 Text Label File (.list): {asr_output_file}")
print(f"🔹 Train Dataset Folder: {sliced_folder}")
print("-" * 60)
print("💡 LƯU Ý QUAN TRỌNG:")
print("1. Khi WebUI mở ra, hãy tích chọn 'v2Pro' ở ngay đầu trang.")
print("2. Mọi dữ liệu training và model (.pth, .ckpt) đã được cấu hình tự động lưu vào Drive.")
print("3. Nếu sập Runtime, chỉ cần chạy lại B1, B2 và B5 để Train tiếp.")
print("="*60)

# 4. Vá lỗi chia sẻ link và chạy WebUI
!sed -i "s/is_share = False/is_share = True/g" config.py
# Chạy với ngôn ngữ Tiếng Việt (nếu repo hỗ trợ)
!python webui.py --share

⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: Giong_Doc_Sach_02
/content/GPT-SoVITS-Vietnamese
🚀 ĐANG KHỞI ĐỘNG GPT-SOVITS WebUI...
📋 THÔNG SỐ ĐỂ ĐIỀN VÀO WEBUI (TAB 1A & 1B):
🔹 Experiment Name: Giong_Doc_Sach_02
🔹 Text Label File (.list): /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/Giong_Doc_Sach_02.list
🔹 Train Dataset Folder: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/sliced_audio
------------------------------------------------------------
💡 LƯU Ý QUAN TRỌNG:
1. Khi WebUI mở ra, hãy tích chọn 'v2Pro' ở ngay đầu trang.
2. Mọi dữ liệu training và model (.pth, .ckpt) đã được cấu hình tự động lưu vào Drive.
3. Nếu sập Runtime, chỉ cần chạy lại B1, B2 và B5 để Train tiếp.
Running on local URL:  http://0.0.0.0:9874
Running on public URL: https://7eed485f1d5cf42ad5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
"/usr/bin/pytho

In [ ]:
# @title 3. Tải Dữ liệu Audio
# @markdown Chọn nguồn dữ liệu và upload file. Tự động dùng tên Experiment đã đặt ở Bước 1.

import os
from google.colab import files
import shutil

# 1. Kiểm tra và lấy tên Experiment
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_Default"
    print(f"⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: {exp_name}")
else:
    exp_name = global_exp_name

# 2. Thiết lập đường dẫn
# dataset_root thường được định nghĩa ở B2, nếu chưa có sẽ dùng mặc định /content/dataset
if 'dataset_root' not in globals():
    dataset_root = "/content/dataset"

input_audio_folder = f"{dataset_root}/{exp_name}"
os.makedirs(input_audio_folder, exist_ok=True)

# 3. Cấu hình nguồn tải
source_type = "Direct Upload" # @param ["Direct Upload", "Google Drive"]
# @markdown Nếu chọn Google Drive, hãy nhập đường dẫn thư mục chứa audio của bạn:
google_drive_path = "/content/drive/MyDrive/Data_Audio" # @param {type:"string"}

if source_type == "Google Drive":
    if os.path.exists(google_drive_path):
        print(f"🚀 Đang sao chép audio từ Drive: {google_drive_path}...")
        count = 0
        for f in os.listdir(google_drive_path):
            if f.lower().endswith((".wav", ".mp3", ".flac", ".m4a")):
                shutil.copy(os.path.join(google_drive_path, f), input_audio_folder)
                count += 1
        print(f"✅ Đã sao chép xong {count} file vào {input_audio_folder}")
    else:
        print(f"❌ Lỗi: Không tìm thấy thư mục '{google_drive_path}' trên Drive của bạn!")

else:
    print("📂 Vui lòng chọn các file audio từ máy tính để upload:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest_path = os.path.join(input_audio_folder, filename)
        # Nếu file đã tồn tại thì xóa trước khi di chuyển để tránh lỗi
        if os.path.exists(dest_path):
            os.remove(dest_path)
        shutil.move(filename, dest_path)
    print(f"✅ Đã upload xong vào {input_audio_folder}")

# 4. Dọn dẹp rác & Kiểm tra kết quả
# Xóa các folder ẩn hệ thống nếu có (nguyên nhân gây lỗi ASR)
junk_folder = os.path.join(input_audio_folder, ".ipynb_checkpoints")
if os.path.exists(junk_folder):
    shutil.rmtree(junk_folder)

audio_files = [f for f in os.listdir(input_audio_folder) if f.lower().endswith((".wav", ".mp3", ".flac", ".m4a"))]

print("-" * 30)
print(f"📁 Thư mục lưu trữ: {input_audio_folder}")
print(f"🎵 Tổng cộng: {len(audio_files)} file audio sẵn sàng.")
if len(audio_files) > 0:
    print(f"🔍 File tiêu biểu: {audio_files[0]}")
else:
    print("⚠️ Cảnh báo: Không tìm thấy file audio nào. Vui lòng kiểm tra lại!")

📂 Vui lòng chọn các file audio từ máy tính để upload:


Saving Giong_Doc_Sach_02.mp3 to Giong_Doc_Sach_02.mp3
✅ Đã upload xong vào /content/dataset/Giong_Doc_Sach_02
------------------------------
📁 Thư mục lưu trữ: /content/dataset/Giong_Doc_Sach_02
🎵 Tổng cộng: 1 file audio sẵn sàng.
🔍 File tiêu biểu: Giong_Doc_Sach_02.mp3


In [ ]:
# @title 4. Cắt Audio + Tạo Phụ Đề Tiếng Việt (ASR)
# @markdown Tự động cắt và tạo file list dựa trên tên Experiment. Dữ liệu sẽ được lưu thẳng vào Drive.

import os
import subprocess
import sys

# 1. Kiểm tra và lấy tên Experiment
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_Default"
    print(f"⚠️ Cảnh báo: Chưa chạy Bước 1. Dùng tên mặc định: {exp_name}")
else:
    exp_name = global_exp_name

# 2. Thiết lập đường dẫn (Lưu vào Drive để không phải làm lại nếu sập máy)
%cd /content/GPT-SoVITS-Vietnamese
drive_base = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}"
input_audio_folder = f"/content/dataset/{exp_name}"
sliced_output_folder = f"{drive_base}/sliced_audio"
asr_output_file = f"{drive_base}/{exp_name}.list"

os.makedirs(sliced_output_folder, exist_ok=True)
os.makedirs("output", exist_ok=True)

# --- FIX LỖI HỆ THỐNG ---
print("🛠️ Đang cấu hình môi trường...")
!sed -i 's|cmd=\[r"C:.*ffmpeg-win-x86_64-v7.1.exe", "-nostdin"\]|cmd=["ffmpeg", "-nostdin"]|g' tools/my_utils.py
!pip install -q faster-whisper
sys.path.insert(0, '/content/GPT-SoVITS-Vietnamese')
os.environ['PYTHONPATH'] = '/content/GPT-SoVITS-Vietnamese'

# @markdown ---
# @markdown **Tùy chọn ASR:**
use_uploaded_script = False # @param {type:"boolean"}
# @markdown Chọn 'force_re_run' nếu bạn muốn cắt lại audio từ đầu:
force_re_run = False # @param {type:"boolean"}

# ========== BƯỚC 1: CẮT AUDIO ==========
print("="*60)
print("🔪 BƯỚC 1: CẮT AUDIO THÀNH CÁC ĐOẠN NGẮN")
print("="*60)

# Kiểm tra dữ liệu đầu vào
if not os.path.exists(input_audio_folder) or len(os.listdir(input_audio_folder)) == 0:
    print(f"❌ LỖI: Thư mục {input_audio_folder} rỗng hoặc không tồn tại!")
    print("👉 Vui lòng chạy lại Bước 3 để upload audio.")
else:
    # Chỉ chạy cắt audio nếu thư mục đích trống hoặc chọn force_re_run
    if len(os.listdir(sliced_output_folder)) == 0 or force_re_run:
        print("⏳ Đang tiến hành cắt audio (Slicing)...")
        !rm -rf {sliced_output_folder}/*
        slice_cmd = f'python tools/slice_audio.py "{input_audio_folder}" "{sliced_output_folder}" -34 4000 300 10 500 0.9 0.25 0 1'
        os.system(slice_cmd)
    else:
        print(f"✅ Đã có sẵn audio đã cắt trong Drive ({len(os.listdir(sliced_output_folder))} file). Bỏ qua bước cắt.")

# ========== BƯỚC 2: ASR (NHẬN DẠNG CHỮ) ==========
print("\n" + "="*60)
if use_uploaded_script:
    print("📂 BƯỚC 2: SỬ DỤNG SCRIPT UPLOAD")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        filename = list(uploaded.keys())[0]
        shutil.move(filename, asr_output_file)
        print(f"✅ Đã lưu script vào: {asr_output_file}")
else:
    print("🎤 BƯỚC 2: CHẠY ASR TIẾNG VIỆT (WHISPER LARGE-V3)")
    if os.path.exists(asr_output_file) and not force_re_run:
        print(f"✅ Đã có file phụ đề (.list) trên Drive. Bỏ qua ASR.")
    else:
        print("⏳ Đang nhận diện giọng nói... (Vui lòng chờ)")
        asr_cmd = f'python tools/asr/fasterwhisper_asr.py -i "{sliced_output_folder}" -o "{drive_base}" -s large-v3 -l vi -p float16'
        os.system(asr_cmd)

        # Rename file .list cho đúng tên exp_name
        raw_list = f"{drive_base}/sliced_audio.list"
        if os.path.exists(raw_list):
            if os.path.exists(asr_output_file): os.remove(asr_output_file)
            os.rename(raw_list, asr_output_file)

# ========== TỔNG KẾT ==========
if os.path.exists(asr_output_file):
    with open(asr_output_file, "r", encoding="utf-8") as f:
        lines = f.readlines()
    print("\n✅ THÀNH CÔNG!")
    print(f"📄 File Label (.list) nằm tại: {asr_output_file}")
    print(f"📝 Tổng số câu thoại: {len(lines)}")
    print(f"📁 Audio đã cắt nằm tại: {sliced_output_folder}")
    print("\n👉 BÂY GIỜ BẠN CÓ THỂ MỞ WEBUI VÀ SỬ DỤNG ĐƯỜNG DẪN .LIST TRÊN ĐỂ TRAIN.")
else:
    print("❌ LỖI: Không tìm thấy hoặc không tạo được file .list")

/content/GPT-SoVITS-Vietnamese
🛠️ Đang cấu hình môi trường...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 100.8 MB/s eta 0:00:00
🔪 BƯỚC 1: CẮT AUDIO THÀNH CÁC ĐOẠN NGẮN
⏳ Đang tiến hành cắt audio (Slicing)...

🎤 BƯỚC 2: CHẠY ASR TIẾNG VIỆT (WHISPER LARGE-V3)
⏳ Đang nhận diện giọng nói... (Vui lòng chờ)

✅ THÀNH CÔNG!
📄 File Label (.list) nằm tại: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/Giong_Doc_Sach_02.list
📝 Tổng số câu thoại: 29
📁 Audio đã cắt nằm tại: /content/drive/MyDrive/GPT_SoVITS_Data/Giong_Doc_Sach_02/sliced_audio

👉 BÂY GIỜ BẠN CÓ THỂ MỞ WEBUI VÀ SỬ DỤNG ĐƯỜNG DẪN .LIST TRÊN ĐỂ TRAIN.


In [ ]:
import os

def ultimate_fix():
    # 1. Sửa lỗi bảo mật PyTorch 2.6
    files_to_patch = [
        "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/prepare_datasets/2-get-sv.py",
        "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/AR/data/data_module.py"
    ]

    # 2. Sửa lỗi bộ lọc dữ liệu và lỗi chia cho 0
    logic_files = [
        "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/AR/data/dataset.py",
        "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/module/data_utils.py"
    ]

    for fpath in files_to_patch + logic_files:
        if os.path.exists(fpath):
            with open(fpath, 'r') as f:
                c = f.read()

            # Fix nạp file cũ
            c = c.replace('torch.load(sv_path, map_location="cpu")', 'torch.load(sv_path, map_location="cpu", weights_only=False)')
            c = c.replace('torch.load(dict_s1, map_location="cpu")', 'torch.load(dict_s1, map_location="cpu", weights_only=False)')

            # Fix nới lỏng bộ lọc tốc độ nói (Filter)
            c = c.replace('if(norm_text_len/wav_dur<3 or norm_text_len/wav_dur>25):continue', 'if(norm_text_len/wav_dur<0.01 or norm_text_len/wav_dur>100):continue')
            c = c.replace('if phoneme_len / semantic_len > 25 or phoneme_len / semantic_len < 3:', 'if phoneme_len / semantic_len > 100 or phoneme_len / semantic_len < 0.01:')

            # Fix lỗi chia cho 0
            c = c.replace('int(min_num / leng)', 'int(min_num / max(leng, 1))')

            # Fix lỗi Assertion ít nhất 1 file
            c = c.replace('assert len(audiopaths_sid_text_new) > 1', 'assert len(audiopaths_sid_text_new) >= 1')

            with open(fpath, 'w') as f:
                f.write(c)
            print(f"✅ Đã vá lỗi triệt để tại: {fpath}")

ultimate_fix()

✅ Đã vá lỗi triệt để tại: /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/prepare_datasets/2-get-sv.py
✅ Đã vá lỗi triệt để tại: /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/AR/data/data_module.py
✅ Đã vá lỗi triệt để tại: /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/AR/data/dataset.py
✅ Đã vá lỗi triệt để tại: /content/GPT-SoVITS-Vietnamese/GPT_SoVITS/module/data_utils.py


In [ ]:
# @title (Tùy chọn) Chạy lại ASR thủ công
# @markdown Chỉ chạy cell này nếu bạn muốn chạy lại bước tạo phụ đề cho file audio đã cắt.

import os
if 'exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_01"

sliced_audio_folder = "output/slicer_opt"
output_list_file = f"output/{exp_name}.list"

if not os.path.exists(sliced_audio_folder):
    print("❌ LỖI: Không tìm thấy thư mục đã cắt!")
else:
    print("🚀 Đang chạy Whisper (large-v3)...")
    !python tools/asr/fasterwhisper_asr.py -i "{sliced_audio_folder}" -o "output" -s large-v3 -l vi -p float16

    # Rename
    generated = "output/slicer_opt.list"
    if os.path.exists(generated):
        import shutil
        shutil.move(generated, output_list_file)

    print(f"✅ Đã xong: {os.path.abspath(output_list_file)}")

In [ ]:
# Sửa file 2-get-sv.py để bỏ qua kiểm tra bảo mật của PyTorch 2.6
import os

file_path = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/prepare_datasets/2-get-sv.py"
with open(file_path, 'r') as f:
    content = f.read()

# Thay thế lệnh torch.load cũ bằng lệnh cho phép weights_only=False
new_content = content.replace(
    'pretrained_state = torch.load(sv_path, map_location="cpu")',
    'pretrained_state = torch.load(sv_path, map_location="cpu", weights_only=False)'
)

with open(file_path, 'w') as f:
    f.write(new_content)

print("Đã sửa lỗi bảo mật PyTorch. Giờ bạn hãy nhấn lại nút trích xuất trên WebUI nhé!")

Đã sửa lỗi bảo mật PyTorch. Giờ bạn hãy nhấn lại nút trích xuất trên WebUI nhé!


In [ ]:
# @title 6. Kiểm tra & Lưu trữ Model Vĩnh viễn (Final Sync)
# @markdown Cell này giúp bạn kiểm tra lại các file đã train và đảm bảo chúng an toàn trên Drive.

import os
import shutil

# 1. Lấy tên Experiment
if 'global_exp_name' not in globals():
    exp_name = "Giong_Doc_Sach_02"
else:
    exp_name = global_exp_name

# 2. Thiết lập đường dẫn lưu trữ
# Chúng ta lưu vào folder Weights riêng để sau này dễ tìm kiếm
final_drive_path = f"/content/drive/MyDrive/GPT_SoVITS_Data/{exp_name}/FINAL_MODELS"
os.makedirs(final_drive_path, exist_ok=True)

print(f"🕵️ Đang kiểm tra thành quả của Experiment: {exp_name}...")

# 3. Gom các file weights (.pth và .ckpt)
# Vì đã dùng link ảo nên weights_dir thực tế trỏ về Drive
weights_dir = "/content/GPT-SoVITS-Vietnamese/GPT_SoVITS/weights"

if os.path.exists(weights_dir):
    trained_files = [f for f in os.listdir(weights_dir) if f.endswith(('.pth', '.ckpt'))]

    if len(trained_files) > 0:
        print(f"✅ Tìm thấy {len(trained_files)} file model đã huấn luyện.")

        # Sao chép thêm một bản vào thư mục FINAL_MODELS cho chắc chắn
        for file in trained_files:
            src = os.path.join(weights_dir, file)
            dst = os.path.join(final_drive_path, file)
            shutil.copy2(src, dst)
            print(f"   -> Đã sao lưu: {file}")

        print(f"\n🎉 CHÚC MỪNG! Model của bạn đã được bảo vệ vĩnh viễn tại:")
        print(f"📂 {final_drive_path}")
    else:
        print("⚠️ Cảnh báo: Không tìm thấy file model mới nào trong thư mục weights.")
        print("👉 Hãy đảm bảo bạn đã nhấn nút 'Start Training' trong WebUI và đã hoàn thành ít nhất 1 Epoch.")
else:
    print("❌ Lỗi: Thư mục weights không tồn tại.")

# 4. Hiển thị dung lượng đã sử dụng trên Drive (Tùy chọn)
!du -sh "{final_drive_path}"